# ARC Prize 2026 - Minimal Performance Patch Baseline (LB 33.89)

This notebook implements the high-performance ARC-AGI-2 baseline:
- 4x L4 GPU multiprocessing with Unsloth fast fine-tuning.
- Single-task test-time adaptation (TTFT) with LoRA.
- 13-token vocabulary shrinking (digits '0'..'9', '\n', start/end tokens).
- Group equivariant D4 x S10 dihedral and color permutation augmentation.
- Turbo DFS beam search with teacher-forced NLL scoring.
- Inverse-consensus candidate aggregation (score_kgmon).
- Output: Standardized submission.json formatted for ARC Prize 2026.


In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600


In [ ]:
!pip uninstall -y tensorflow


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer
from typing import List, Tuple, Dict, Any, Optional, Set


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation) == list(range(10))
    a = np.asarray(a)
    if a.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim == 2
        if invert:
            permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:
    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception:
            pass
        return None


class ArcDataset:
    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None:
            return a
        for op in key.split('.')[1:]:
            if op == 'rot90':
                a = np.rot90(a)
            elif op == 'transpose':
                a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'):
                a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):
                a = np.copy(a)
            elif op.startswith('out') or op.startswith('ex') or op.startswith('run') or op.startswith('base'):
                a = a
            else:
                raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None:
            return a
        for op in key.split('.')[1:][::-1]:
            if op == 'rot90':
                a = np.rot90(a, k=3)
            elif op == 'transpose':
                a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'):
                a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):
                a = np.copy(a)
            elif op.startswith('out') or op.startswith('ex') or op.startswith('run') or op.startswith('base'):
                a = a
            else:
                raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None:
            keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(queries=json.loads(queries), is_orig=True, keys=keys)

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f:
            replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys=[k for d in datasets for k in d.keys],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t == 'input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None:
            stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train + query + reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train + query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if name == 'input':
                return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name == 'reply':
                return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else:
                assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None:
                    self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len < temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f"ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}"])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k == 'train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None:
                p = p[:keep_max]
            new_key = f"{key}.ex" + ('-' if (p.max() > 9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k == 'train' else v) for k, v in self.queries[key].items()}
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig == True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None:
            self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f"*** Generating submission for {len(results)} outputs...")
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = np.asarray(g, dtype=int).tolist()

    def validate_submission(self, submission):
        assert self.is_orig == True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            correct_attempt_1 = True
            correct_attempt_2 = True
            for i, r in enumerate(v):
                if not np.array_equal(r, submission[k][i]['attempt_1']):
                    correct_attempt_1 = False
                if not np.array_equal(r, submission[k][i]['attempt_2']):
                    correct_attempt_2 = False
            if correct_attempt_1 or correct_attempt_2:
                score += 1
        return score


In [ ]:
%%writefile varc_engine.py
import math
from typing import Optional, Tuple, List, Dict, Any
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    d = x.shape[-1]
    x1 = x[..., : d // 2]
    x2 = x[..., d // 2 :]
    return torch.cat((-x2, x1), dim=-1)


class VisionRotaryEmbeddingFast(nn.Module):
    def __init__(self, dim: int, pt_seq_len: int = 16, theta: float = 10000.0, no_rope: int = 1) -> None:
        super().__init__()
        self.dim = dim
        self.no_rope = no_rope
        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
        t = torch.arange(pt_seq_len).float()
        
        freqs_h = torch.outer(t, freqs)
        freqs_h = torch.repeat_interleave(freqs_h, 2, dim=-1)
        freqs_w = torch.outer(t, freqs)
        freqs_w = torch.repeat_interleave(freqs_w, 2, dim=-1)
        
        H, W = pt_seq_len, pt_seq_len
        fh = freqs_h.unsqueeze(1).expand(H, W, -1)
        fw = freqs_w.unsqueeze(0).expand(H, W, -1)
        freqs_2d = torch.cat((fh, fw), dim=-1)
        freqs_flat = freqs_2d.reshape(H * W, -1)
        
        self.register_buffer("freqs_cos", freqs_flat.cos())
        self.register_buffer("freqs_sin", freqs_flat.sin())

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        seq_len = t.shape[2]
        if self.no_rope > 0:
            prefix = t[:, :, :self.no_rope, :]
            patches = t[:, :, self.no_rope:, :]
            p_len = patches.shape[2]
            cos = self.freqs_cos[:p_len, :].unsqueeze(0).unsqueeze(0)
            sin = self.freqs_sin[:p_len, :].unsqueeze(0).unsqueeze(0)
            patches_rot = (patches * cos) + (rotate_half(patches) * sin)
            return torch.cat((prefix, patches_rot), dim=2)
        else:
            cos = self.freqs_cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
            sin = self.freqs_sin[:seq_len, :].unsqueeze(0).unsqueeze(0)
            return (t * cos) + (rotate_half(t) * sin)


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int = 30, patch_size: int = 2, in_chans: int = 256, embed_dim: int = 256) -> None:
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x).flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim: int = 256, num_heads: int = 8, max_seq_len: int = 226, dropout: float = 0.1, no_rope: int = 1) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        half_head_dim = self.head_dim // 2
        self.rotary = VisionRotaryEmbeddingFast(dim=half_head_dim, pt_seq_len=int(max_seq_len ** 0.5) + 4, no_rope=no_rope)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv(x).view(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = self.rotary(qkv[0]), self.rotary(qkv[1]), qkv[2]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].to(dtype=torch.bool)
            attn_scores = attn_scores.masked_fill(mask, torch.finfo(attn_scores.dtype).min)
        attn_weights = self.attn_dropout(torch.softmax(attn_scores, dim=-1))
        context = torch.matmul(attn_weights, v).transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        return self.proj_dropout(self.proj(context))


class ARCTransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim: int = 256, num_heads: int = 8, mlp_dim: int = 512, dropout: float = 0.1, max_seq_len: int = 226, no_rope: int = 1) -> None:
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(embed_dim=embed_dim, num_heads=num_heads, max_seq_len=max_seq_len, dropout=dropout, no_rope=no_rope)
        self.dropout1, self.norm1 = nn.Dropout(dropout), nn.LayerNorm(embed_dim)
        self.linear1, self.activation = nn.Linear(embed_dim, mlp_dim), nn.GELU()
        self.dropout2, self.linear2 = nn.Dropout(dropout), nn.Linear(mlp_dim, embed_dim)
        self.dropout3, self.norm2 = nn.Dropout(dropout), nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        res = x
        x = self.norm1(res + self.dropout1(self.self_attn(x, key_padding_mask=key_padding_mask)))
        res = x
        x = self.norm2(res + self.dropout3(self.linear2(self.dropout2(self.activation(self.linear1(x))))))
        return x


class ARCViT(nn.Module):
    def __init__(self, num_tasks: int = 400, image_size: int = 30, num_colors: int = 11, embed_dim: int = 256, depth: int = 6, num_heads: int = 8, mlp_dim: int = 512, dropout: float = 0.1, num_task_tokens: int = 1, patch_size: int = 2) -> None:
        super().__init__()
        self.image_size, self.num_colors, self.embed_dim, self.patch_size = image_size, num_colors, embed_dim, patch_size
        self.seq_length = (image_size // patch_size) ** 2
        self.num_task_tokens = num_task_tokens
        self.color_embed = nn.Embedding(num_colors, embed_dim)
        self.task_token_embed = nn.Embedding(num_tasks, embed_dim * num_task_tokens)
        self.patch_embed = PatchEmbed(image_size, patch_size, embed_dim, embed_dim)
        total_seq_len = num_task_tokens + self.seq_length
        self.positional_embed = nn.Parameter(torch.zeros(1, self.seq_length, embed_dim))
        self.encoder = nn.ModuleList([ARCTransformerEncoderLayer(embed_dim=embed_dim, num_heads=num_heads, mlp_dim=mlp_dim, dropout=dropout, max_seq_len=total_seq_len, no_rope=num_task_tokens) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_colors * (patch_size ** 2))
        self._reset_parameters()

    def _reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.positional_embed, std=0.02)
        nn.init.trunc_normal_(self.task_token_embed.weight, std=0.02)
        nn.init.trunc_normal_(self.color_embed.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, pixel_values: torch.Tensor, task_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size = pixel_values.size(0)
        device = pixel_values.device
        tokens = self.color_embed(pixel_values.long())
        tokens = self.patch_embed(tokens.permute(0, 3, 1, 2)) + self.positional_embed[:, : self.seq_length, :]
        task_tokens = self.task_token_embed(task_ids.long()).reshape(batch_size, self.num_task_tokens, -1)
        hidden_states = torch.cat([task_tokens, tokens], dim=1)

        key_padding_mask = None
        if attention_mask is not None:
            mask_patches = attention_mask.reshape(batch_size, self.image_size // self.patch_size, self.patch_size, self.image_size // self.patch_size, self.patch_size)
            mask_patches = mask_patches.amax(dim=(2, 4)).reshape(batch_size, self.seq_length)
            key_padding_mask = torch.cat([torch.zeros(batch_size, self.num_task_tokens, device=device, dtype=torch.bool), ~mask_patches.bool()], dim=1)

        for layer in self.encoder:
            hidden_states = layer(hidden_states, key_padding_mask=key_padding_mask)
        pixel_states = self.norm(hidden_states)[:, self.num_task_tokens:, :]
        logits = self.head(pixel_states)
        G, P = self.image_size // self.patch_size, self.patch_size
        logits = logits.reshape(batch_size, G, G, P, P, self.num_colors).permute(0, 1, 3, 2, 4, 5).reshape(batch_size, self.image_size, self.image_size, self.num_colors)
        return logits.permute(0, 3, 1, 2)


class VARCCanvasProcessor:
    def __init__(self, canvas_size: int = 30, pad_val: int = 10) -> None:
        self.canvas_size, self.pad_val = canvas_size, pad_val

    def grid_to_canvas(self, grid: np.ndarray, offset_r: int = 0, offset_c: int = 0) -> Tuple[np.ndarray, np.ndarray]:
        H, W = grid.shape
        canvas = np.full((self.canvas_size, self.canvas_size), self.pad_val, dtype=np.int64)
        mask = np.zeros((self.canvas_size, self.canvas_size), dtype=bool)
        r_start = max(0, min(offset_r, self.canvas_size - H))
        c_start = max(0, min(offset_c, self.canvas_size - W))
        canvas[r_start:r_start + H, c_start:c_start + W] = grid
        mask[r_start:r_start + H, c_start:c_start + W] = True
        return canvas, mask

    def canvas_to_grid(self, canvas: np.ndarray, target_h: int, target_w: int, offset_r: int = 0, offset_c: int = 0) -> np.ndarray:
        r_start = max(0, min(offset_r, self.canvas_size - target_h))
        c_start = max(0, min(offset_c, self.canvas_size - target_w))
        extracted = canvas[r_start:r_start + target_h, c_start:c_start + target_w]
        return np.where(extracted == self.pad_val, 0, extracted).astype(int)


class VARCSolver:
    def __init__(self, model_weights_path: Optional[str] = None, device: str = "cpu", canvas_size: int = 30, ttt_steps: int = 60, lr: float = 3e-4) -> None:
        self.device = torch.device(device)
        self.canvas_size, self.ttt_steps, self.lr = canvas_size, ttt_steps, lr
        self.processor = VARCCanvasProcessor(canvas_size=canvas_size)
        self.model = ARCViT(num_tasks=1, image_size=canvas_size, num_colors=11).to(self.device)

    def solve_task(self, train_pairs: List[Dict[str, np.ndarray]], test_input: np.ndarray, target_shape: Optional[Tuple[int, int]] = None) -> np.ndarray:
        if not target_shape:
            out_shapes = [p["output"].shape for p in train_pairs]
            target_shape = out_shapes[0] if len(set(out_shapes)) == 1 else test_input.shape

        task_model = ARCViT(num_tasks=1, image_size=self.canvas_size, num_colors=11).to(self.device)
        task_model.load_state_dict(self.model.state_dict())
        task_model.train()
        optimizer = torch.optim.AdamW(task_model.parameters(), lr=self.lr, weight_decay=1e-4)

        inp_tensors, tgt_tensors, mask_tensors = [], [], []
        for pair in train_pairs:
            inp_g, tgt_g = np.asarray(pair["input"]), np.asarray(pair["output"])
            for (dr, dc) in [(0, 0), (2, 2), (0, 4), (4, 0)]:
                c_inp, m_inp = self.processor.grid_to_canvas(inp_g, offset_r=dr, offset_c=dc)
                c_tgt, _ = self.processor.grid_to_canvas(tgt_g, offset_r=dr, offset_c=dc)
                inp_tensors.append(torch.tensor(c_inp, dtype=torch.long))
                tgt_tensors.append(torch.tensor(c_tgt, dtype=torch.long))
                mask_tensors.append(torch.tensor(m_inp, dtype=torch.bool))

        if not inp_tensors:
            return np.zeros(target_shape, dtype=int)

        batch_inp = torch.stack(inp_tensors).to(self.device)
        batch_tgt = torch.stack(tgt_tensors).to(self.device)
        batch_mask = torch.stack(mask_tensors).to(self.device)
        task_ids = torch.zeros(len(inp_tensors), dtype=torch.long, device=self.device)

        for _ in range(self.ttt_steps):
            optimizer.zero_grad(set_to_none=True)
            logits = task_model(batch_inp, task_ids, attention_mask=batch_mask)
            loss = F.cross_entropy(logits, batch_tgt, ignore_index=10)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(task_model.parameters(), 1.0)
            optimizer.step()

        task_model.eval()
        predictions = []
        with torch.no_grad():
            for (dr, dc) in [(0, 0), (1, 1), (2, 0), (0, 2)]:
                t_canvas, t_mask = self.processor.grid_to_canvas(test_input, offset_r=dr, offset_c=dc)
                t_inp = torch.tensor(t_canvas, dtype=torch.long, device=self.device).unsqueeze(0)
                t_m = torch.tensor(t_mask, dtype=torch.bool, device=self.device).unsqueeze(0)
                out_logits = task_model(t_inp, torch.zeros(1, dtype=torch.long, device=self.device), attention_mask=t_m)
                pred_canvas = out_logits.argmax(dim=1).squeeze(0).cpu().numpy()
                pred_grid = self.processor.canvas_to_grid(pred_canvas, target_h=target_shape[0], target_w=target_shape[1], offset_r=dr, offset_c=dc)
                predictions.append(pred_grid)

        hashes = [tuple(map(tuple, p)) for p in predictions]
        best_hash = max(set(hashes), key=hashes.count)
        for p in predictions:
            if tuple(map(tuple, p)) == best_hash:
                return p
        return predictions[0]


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import json
import pickle
import numpy as np
from typing import List, Tuple, Dict, Any, Optional

def hashable(guess):
    return tuple(map(tuple, guess))

def hashable_tuple(guesses_list):
    return tuple(tuple(map(tuple, g)) for g in guesses_list)

def score_sum(guesses, getter):
    all_scores = {}
    for subkey, sample in guesses.items():
        solution = hashable(sample["solution"])
        all_scores[solution] = all_scores.get(solution, []) + [sample]
    for key, guesses in all_scores.items():
        all_scores[key] = getter(guesses)
    ordered_outputs = sorted(all_scores.keys(), key=lambda x: all_scores[x], reverse=True)
    ordered_outputs = [np.asarray(x) for x in ordered_outputs]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)

class ArcDecoder:
    def __init__(self, dataset, n_guesses=2):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        self.varc_predictions = {}
        self.raw_shards = {}

    def load_decoded_results(self, store, run_name=""):
        if not os.path.exists(store):
            return
        for filename in os.listdir(store):
            shard_path = os.path.join(store, filename)
            if not os.path.isfile(shard_path):
                continue
            if filename.startswith("varc_"):
                try:
                    with open(shard_path, "r") as f:
                        v_data = json.load(f)
                    self.varc_predictions.update(v_data)
                except Exception as e:
                    print(f"Warning loading varc shard {filename}: {e}")
                continue
            try:
                with bz2.BZ2File(shard_path) as f:
                    outputs = pickle.load(f)
                parts = filename.split(".")
                base_key = parts[0]
                view_tag = ".".join(parts[1:]) if len(parts) > 1 else "default"
                puzzle_id, test_idx = base_key.split("_")
                test_idx = int(test_idx)

                self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
                for i, sample in enumerate(outputs):
                    self.decoded_results[base_key][f"{filename}{run_name}.out{i}"] = sample

                tuple_key = (puzzle_id, view_tag)
                if tuple_key not in self.raw_shards:
                    self.raw_shards[tuple_key] = {}
                self.raw_shards[tuple_key][test_idx] = outputs
            except Exception as e:
                pass

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        puzzles = {}
        for bk in self.decoded_results.keys():
            pid, tid = bk.split("_")
            if pid not in puzzles:
                puzzles[pid] = {}
            puzzles[pid][int(tid)] = bk

        results = {}

        for pid, test_dict in puzzles.items():
            num_tests = len(test_dict)
            sorted_tids = sorted(test_dict.keys())

            if num_tests > 1:
                joint_tuples = {}
                views_for_pid = [v_tag for (p, v_tag) in self.raw_shards.keys() if p == pid]
                for view_tag in set(views_for_pid):
                    shard_dict = self.raw_shards.get((pid, view_tag), {})
                    if all(tid in shard_dict and len(shard_dict[tid]) > 0 for tid in sorted_tids):
                        joint_cand = [shard_dict[tid][0]["solution"] for tid in sorted_tids]
                        if all(isinstance(g, np.ndarray) and g.ndim == 2 and 1 <= g.shape[0] <= 30 and 1 <= g.shape[1] <= 30 for g in joint_cand):
                            t_hash = hashable_tuple(joint_cand)
                            if t_hash not in joint_tuples:
                                joint_tuples[t_hash] = {
                                    "tuple": joint_cand,
                                    "aug_scores": [],
                                    "count": 0
                                }
                            mean_aug = np.mean([np.mean(shard_dict[tid][0]["score_aug"]) for tid in sorted_tids])
                            joint_tuples[t_hash]["count"] += 1
                            joint_tuples[t_hash]["aug_scores"].append(mean_aug)

                ranked_tuples = []
                for t_hash, t_info in joint_tuples.items():
                    kgmon_score = t_info["count"] - np.mean(t_info["aug_scores"])
                    ranked_tuples.append((kgmon_score, t_info["tuple"]))
                ranked_tuples.sort(key=lambda x: x[0], reverse=True)

                if len(ranked_tuples) > 0:
                    attempt_1_tuple = ranked_tuples[0][1]
                    attempt_2_tuple = ranked_tuples[1][1] if len(ranked_tuples) > 1 else attempt_1_tuple
                else:
                    attempt_1_tuple = [selection_algorithm(self.decoded_results[test_dict[tid]])[0] for tid in sorted_tids]
                    attempt_2_tuple = attempt_1_tuple

                varc_valid = True
                varc_tuple = []
                for tid in sorted_tids:
                    bk = test_dict[tid]
                    if bk in self.varc_predictions:
                        v_pred = np.asarray(self.varc_predictions[bk], dtype=int)
                        if v_pred.ndim == 2 and 1 <= v_pred.shape[0] <= 30 and 1 <= v_pred.shape[1] <= 30:
                            varc_tuple.append(v_pred)
                        else:
                            varc_valid = False
                    else:
                        varc_valid = False

                if varc_valid and len(varc_tuple) == num_tests:
                    is_dup = all(np.array_equal(varc_tuple[i], attempt_1_tuple[i]) for i in range(num_tests))
                    if not is_dup:
                        attempt_2_tuple = varc_tuple

                for i, tid in enumerate(sorted_tids):
                    bk = test_dict[tid]
                    results[bk] = [attempt_1_tuple[i], attempt_2_tuple[i]]

            else:
                tid = sorted_tids[0]
                bk = test_dict[tid]
                v = self.decoded_results[bk]
                ordered = selection_algorithm({k: g for k, g in v.items()})
                if not ordered:
                    ordered = [np.zeros((1, 1), dtype=int), np.zeros((1, 1), dtype=int)]

                attempt_1 = ordered[0]
                attempt_2 = ordered[1] if len(ordered) > 1 else attempt_1

                if bk in self.varc_predictions:
                    v_pred = np.asarray(self.varc_predictions[bk], dtype=int)
                    if v_pred.ndim == 2 and 1 <= v_pred.shape[0] <= 30 and 1 <= v_pred.shape[1] <= 30:
                        if not np.array_equal(v_pred, attempt_1):
                            attempt_2 = v_pred

                results[bk] = [attempt_1, attempt_2] + ordered[2:]

        return results

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")
        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0
        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():
            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)
            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():
                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"
                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores) if correct_beam_scores else 0:8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores) if correct_beam_scores else 0:8.5f}")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
from varc_engine import VARCSolver

import gc
import os
import io
import time
import json
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union, Set, Tuple
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10, "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


_ARC_TOKEN_ID_CACHE = {}

def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:
    n = logits.size(0)
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    for i in range(len(batch_tokens)):
        batch_tokens[i] += [tokenizer.pad_token_id] * (max_len - batch_lengths[i])
    batch_tokens = torch.tensor(batch_tokens, device=model.device, dtype=torch.long)
    outputs = model(batch_tokens)
    logits = outputs.logits
    batch_scores = []
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_len = len(query_tokens)
        answer_len = len(answer_tokens)
        target_tokens = batch_tokens[i, query_len:query_len + answer_len]
        target_logits = logits[i, query_len - 1:query_len + answer_len - 1]
        log_probs = torch.log_softmax(target_logits, dim=-1)
        token_log_probs = log_probs.gather(dim=-1, index=target_tokens.unsqueeze(-1)).squeeze(-1)
        score = -token_log_probs.sum().item()
        batch_scores.append(score)
    return batch_scores


def worker(rank, queue, end_time):
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model_path = "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
    if not os.path.exists(model_path):
        model_path = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    # Deep clone initial weights on CPU to avoid in-place tensor reference mutation
    raw_default = get_peft_model_state_dict(model, adapter_name="default")
    default_weights_cpu = {k: v.cpu().clone().detach() for k, v in raw_default.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        if not os.path.exists(test_path):
            test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
        if not os.path.exists(test_path):
            test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)
    dir_outputs = "/tmp/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()
        puzzle_ds = arc_test_set.change_keys([key])

        # Step 1: Run VARC 2D Vision TTT
        try:
            varc_solver = VARCSolver(device=model.device, ttt_steps=60)
            train_pairs = [
                {"input": np.array(p["input"]), "output": np.array(p["output"])}
                for p in puzzle_ds.queries[key]["train"]
            ]
            varc_task_preds = {}
            for test_idx, test_item in enumerate(puzzle_ds.queries[key]["test"]):
                test_in = np.array(test_item["input"])
                v_pred = varc_solver.solve_task(train_pairs, test_in)
                subkey = f"{key}_{test_idx}"
                varc_task_preds[subkey] = v_pred.tolist()

            with open(os.path.join(dir_outputs, f"varc_{key}.json"), "w") as f:
                json.dump(varc_task_preds, f)
            print(f"[Rank {rank}] VARC solved {key} successfully")
        except Exception as e:
            print(f"[Rank {rank}] VARC error for {key}: {e}")

        # Step 2: Run Qwen-4B TTFT (33.89 LB Baseline)
        fresh_weights = {k: v.clone().to(model.device) for k, v in default_weights_cpu.items()}
        set_peft_model_state_dict(
            model,
            fresh_weights,
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)
        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)
        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()
        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
            known_scores = {}
            for subkeys in batches:
                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, b_tokens in scored_beams:
                        array = formatter.convert_tokens_to_array(b_tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                            aug_queries, aug_answers = [], []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import numpy as np
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    from arc_solver import worker
    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    print(f"[Rank {rank}] start!")
    worker(rank, queue, end_time)
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        if not os.path.exists(test_path):
            test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
        if not os.path.exists(test_path):
            test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        raw_tasks = json.load(f)

    queue = mp.Manager().Queue()
    count = 0
    for key in sorted(raw_tasks.keys()):
        if not rerun_mode:
            if count >= 20:
                continue
            count += 1
        queue.put(key)
    for _ in range(4):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
import glob
import shutil
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
if rerun_mode:
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    if not os.path.exists(test_path):
        test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    data = ArcDataset.from_file(test_path)
else:
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    if not os.path.exists(test_path):
        test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    sol_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
    if not os.path.exists(sol_path):
        sol_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
    data = ArcDataset.from_file(test_path)
    data = data.load_replies(sol_path)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/tmp/inference_outputs")

if not rerun_mode:
    decoder.benchmark_selection_algos()

gpu_results = decoder.run_selection_algo()
submission = data.get_submission(gpu_results)

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Final Hybrid Score:", data.validate_submission(reload_submission))

assert os.path.exists("submission.json"), "Error: submission.json was not generated!"
with open("submission.json", "r") as f:
    sub_data = json.load(f)
assert len(sub_data) > 0, "Error: submission.json is empty!"

print(f"Submission verification successful: {len(sub_data)} tasks formatted.")

files_to_clean = ["arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py", "varc_engine.py"]
for f_path in files_to_clean:
    if os.path.exists(f_path):
        os.remove(f_path)
        print(f"Removed helper script: {f_path}")

for lock_file in glob.glob("/kaggle/worker*"):
    try: os.remove(lock_file)
    except: pass

if os.path.exists("__pycache__"):
    shutil.rmtree("__pycache__", ignore_errors=True)
if os.path.exists("unsloth_compiled_cache"):
    shutil.rmtree("unsloth_compiled_cache", ignore_errors=True)

remaining = os.listdir(".")
print("Final Working Directory contents:", remaining)
assert os.path.exists("submission.json"), "Error: submission.json missing after cleanup!"
assert os.path.getsize("submission.json") > 0, "Error: submission.json is 0 bytes!"
assert not any(f.endswith(".py") for f in remaining), "Working directory contains remaining python scripts!"


# Remove intermediate helper scripts so ONLY submission.json remains as the output artifact
for temp_file in ["arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py"]:
    if os.path.exists(temp_file):
        try:
            os.remove(temp_file)
        except Exception:
            pass
